# ArmanNN Training on Google Colab

This notebook trains ArmanNN on the Stanford IMDB dataset using a free Colab GPU.

**Instructions:**
1. Go to Runtime → Change runtime type → Select **T4 GPU**
2. Run all cells in order

## 1. Clone the repo and install dependencies

In [ ]:
# Clone your repo (replace with your actual repo URL if hosted on GitHub)
# If uploading manually, skip this cell and upload the project folder instead

# Option A: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/Arman-NN.git
# %cd Arman-NN

# Option B: Upload from local (run this, then use the file upload widget)
import os
if not os.path.exists('Arman-NN'):
    print("Upload your Arman-NN project as a zip file:")
    from google.colab import files
    uploaded = files.upload()
    !unzip -q *.zip -d Arman-NN 2>/dev/null || unzip -q *.zip

%cd Arman-NN

In [ ]:
!pip install -q torch datasets transformers numpy

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Load the dataset

In [ ]:
import sys
sys.path.insert(0, '.')

from arman.training.data import load_dataset_from_source

# Load full IMDB training set
train_dataset = load_dataset_from_source(
    source="huggingface",
    dataset_name="stanfordnlp/imdb",
    tokenizer_name="gpt2",
    seq_len=512,
    split="train",
    text_column="text",
    max_samples=0,  # 0 = all 25,000 reviews
)

# Load eval split
eval_dataset = load_dataset_from_source(
    source="huggingface",
    dataset_name="stanfordnlp/imdb",
    tokenizer_name="gpt2",
    seq_len=512,
    split="test",
    text_column="text",
    max_samples=2000,  # Subset for faster eval
)

print(f"Train sequences: {len(train_dataset)}")
print(f"Eval sequences: {len(eval_dataset)}")

## 3. Configure and build the model

In [ ]:
from arman.model import ArmanConfig, ArmanNN

config = ArmanConfig(
    vocab_size=50257,       # GPT-2 tokenizer vocab
    d_model=256,
    n_layers=6,
    n_heads=8,
    max_seq_len=512,
    mlp_hidden=1024,
    expert_hidden=1024,
    n_experts=4,
    moe_top_k=2,
    ssm_state_size=64,
    memory_slots=64,
    graph_layers=2,
)

model = ArmanNN(config)
print(f"Parameters: {model.parameter_count():,}")

## 4. Train with the Trainer

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

from arman.training import Trainer, TrainConfig

train_config = TrainConfig(
    # Optimization
    learning_rate=3e-4,
    weight_decay=0.1,
    max_grad_norm=1.0,
    batch_size=16,
    gradient_accumulation_steps=2,  # Effective batch = 32

    # Schedule
    warmup_steps=200,
    total_steps=5000,
    min_lr_ratio=0.1,

    # Mixed precision (big speedup on T4)
    use_amp=True,
    amp_dtype="float16",  # T4 doesn't support bf16

    # Distributed (single GPU on Colab)
    parallel_mode="none",

    # Checkpointing
    checkpoint_dir="checkpoints",
    save_every_steps=500,
    resume=True,

    # Logging
    log_every_steps=50,
    eval_every_steps=500,
)

trainer = Trainer(
    model_config=config,
    train_config=train_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("Starting training...")
trainer.train()

## 5. Evaluate the trained model

In [ ]:
from arman.training import Evaluator, EvalConfig

eval_config = EvalConfig(batch_size=16, use_amp=True, amp_dtype="float16")
evaluator = Evaluator(model=trainer.model, eval_config=eval_config, device=trainer.device)

metrics = evaluator.evaluate(eval_dataset)
print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  Loss:           {metrics.loss:.4f}")
print(f"  Perplexity:     {metrics.perplexity:.2f}")
print(f"  Top-1 Accuracy: {metrics.top1_accuracy*100:.2f}%")
print(f"  Top-5 Accuracy: {metrics.top5_accuracy*100:.2f}%")
print(f"  MRR:            {metrics.mrr:.4f}")
print("=" * 50)

## 6. Generate text from the trained model

In [ ]:
from transformers import AutoTokenizer
from generate import generate

tokenizer = AutoTokenizer.from_pretrained("gpt2")
device = trainer.device

# Encode a prompt
prompt = "This movie was absolutely"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# Generate
trainer.model.eval()
output_ids = generate(
    trainer.model,
    input_ids,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    top_p=0.9,
)

# Decode
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Generated: {generated_text}")

## 7. Save checkpoint to Google Drive (optional)

In [ ]:
# Mount Google Drive and copy checkpoint for persistence
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ArmanNN
!cp -r checkpoints/ /content/drive/MyDrive/ArmanNN/
print("Checkpoints saved to Google Drive!")